<a href="https://colab.research.google.com/github/mscids2024pranita-hue/goa-rainfall-arima/blob/main/01_data_loading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Data Loading
**Project:** Goa Rainfall Forecasting using SARIMA
**Data:** IMD Gridded Daily Rainfall (0.25°), 1991–2025
**Step:** Install libraries, load one NetCDF file, understand its structure

In [4]:
# Install libraries not available by default in Colab
!pip install netCDF4 xarray -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 75.2 MB/s eta 0:00:00


In [5]:
# Import all libraries we need for data loading
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob

print("All libraries imported successfully!")
print(f"xarray version  : {xr.__version__}")
print(f"numpy version   : {np.__version__}")
print(f"pandas version  : {pd.__version__}")

All libraries imported successfully!
xarray version  : 2025.12.0
numpy version   : 2.0.2
pandas version  : 2.2.2


In [6]:
from google.colab import files

uploaded = files.upload()
# A file picker will appear — select your zip file

Saving imd_rainfall-20260513T063252Z-3-001-20260530T092135Z-3-001.zip to imd_rainfall-20260513T063252Z-3-001-20260530T092135Z-3-001.zip


In [7]:
import zipfile
import os

zip_path = "imd_rainfall-20260513T063252Z-3-001-20260530T092135Z-3-001.zip"

# Unzip into a folder called 'imd_data'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("imd_data")

print("Unzipped successfully!")
print("\nContents of imd_data folder:")
for f in sorted(os.listdir("imd_data/imd_rainfall-20260513T063252Z-3-001/imd_rainfall/")):
    print(f"  {f}")

Unzipped successfully!

Contents of imd_data folder:
  RF25_ind1991_rfp25.nc
  RF25_ind1992_rfp25.nc
  RF25_ind1993_rfp25.nc
  RF25_ind1994_rfp25.nc
  RF25_ind1995_rfp25.nc
  RF25_ind1996_rfp25.nc
  RF25_ind1997_rfp25.nc
  RF25_ind1998_rfp25.nc
  RF25_ind1999_rfp25.nc
  RF25_ind2000_rfp25.nc
  RF25_ind2001_rfp25.nc
  RF25_ind2002_rfp25.nc
  RF25_ind2003_rfp25.nc
  RF25_ind2004_rfp25.nc
  RF25_ind2005_rfp25 (1).nc
  RF25_ind2006_rfp25.nc
  RF25_ind2007_rfp25.nc
  RF25_ind2008_rfp25.nc
  RF25_ind2009_rfp25.nc
  RF25_ind2010_rfp25.nc
  RF25_ind2011_rfp25.nc
  RF25_ind2012_rfp25.nc
  RF25_ind2013_rfp25.nc
  RF25_ind2014_rfp25.nc
  RF25_ind2015_rfp25.nc
  RF25_ind2016_rfp25.nc
  RF25_ind2017_rfp25.nc
  RF25_ind2018_rfp25.nc
  RF25_ind2019_rfp25.nc
  RF25_ind2020_rfp25.nc
  RF25_ind2021_rfp25.nc
  RF25_ind2022_rfp25.nc
  RF25_ind2023_rfp25.nc
  RF25_ind2024_rfp25.nc
  RF25_ind2025_rfp25.nc
  RF25_ind_rfp25 (1).nc
  RF25_ind_rfp25 (2).nc
  RF25_ind_rfp25 (3).nc
  RF25_ind_rfp25.nc


In [8]:
# Define the path to the data folder
DATA_DIR = "imd_data/imd_rainfall-20260513T063252Z-3-001/imd_rainfall/"

# Open just one file — 1991
ds = xr.open_dataset(DATA_DIR + "RF25_ind1991_rfp25.nc")

# Print the full structure
print(ds)

<xarray.Dataset> Size: 25MB
Dimensions:    (TIME: 365, LATITUDE: 129, LONGITUDE: 135)
Coordinates:
  * TIME       (TIME) datetime64[ns] 3kB 1991-01-01 1991-01-02 ... 1991-12-31
  * LATITUDE   (LATITUDE) float64 1kB 6.5 6.75 7.0 7.25 ... 38.0 38.25 38.5
  * LONGITUDE  (LONGITUDE) float64 1kB 66.5 66.75 67.0 ... 99.5 99.75 100.0
Data variables:
    RAINFALL   (TIME, LATITUDE, LONGITUDE) float32 25MB ...
Attributes:
    history:      FERRET V6.82   20-Feb-26
    Conventions:  CF-1.0


In [9]:
# Look at just the RAINFALL variable in detail
print("Shape  :", ds['RAINFALL'].shape)
print("Dims   :", ds['RAINFALL'].dims)
print("Dtype  :", ds['RAINFALL'].dtype)
print("Min    :", float(ds['RAINFALL'].min()))
print("Max    :", float(ds['RAINFALL'].max()))
print("NaN count:", int(ds['RAINFALL'].isnull().sum()))

Shape  : (365, 129, 135)
Dims   : ('TIME', 'LATITUDE', 'LONGITUDE')
Dtype  : float32
Min    : 0.0
Max    : 747.7235717773438
NaN count: 4544615


In [10]:
# Goa bounding box
GOA_LAT_MIN, GOA_LAT_MAX = 14.9, 15.7
GOA_LON_MIN, GOA_LON_MAX = 73.7, 74.3

# Slice the dataset to Goa region only
goa = ds['RAINFALL'].sel(
    LATITUDE  = slice(GOA_LAT_MIN, GOA_LAT_MAX),
    LONGITUDE = slice(GOA_LON_MIN, GOA_LON_MAX)
)

print("Goa slice shape:", goa.shape)
print("Latitude points :", goa.LATITUDE.values)
print("Longitude points:", goa.LONGITUDE.values)
print("\nFirst 10 days of spatially averaged Goa rainfall (mm/day):")

# Average across all Goa grid points (ignoring NaN = ocean pixels)
goa_daily = goa.mean(dim=['LATITUDE','LONGITUDE'], skipna=True)
print(goa_daily.values[:10])

Goa slice shape: (365, 3, 3)
Latitude points : [15.   15.25 15.5 ]
Longitude points: [73.75 74.   74.25]

First 10 days of spatially averaged Goa rainfall (mm/day):
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [11]:
# Check July 1991 — should show heavy rainfall
july_mask = ds['TIME'].dt.month == 7
goa_july = goa_daily.sel(TIME=july_mask)

print("July 1991 daily rainfall (mm/day):")
for date, val in zip(goa_july.TIME.values, goa_july.values):
    print(f"  {str(date)[:10]}  :  {val:.1f} mm")

print(f"\nJuly 1991 total : {float(goa_july.sum()):.1f} mm")
print(f"July 1991 max   : {float(goa_july.max()):.1f} mm")

July 1991 daily rainfall (mm/day):
  1991-07-01  :  16.7 mm
  1991-07-02  :  24.1 mm
  1991-07-03  :  23.4 mm
  1991-07-04  :  27.5 mm
  1991-07-05  :  45.5 mm
  1991-07-06  :  30.7 mm
  1991-07-07  :  63.6 mm
  1991-07-08  :  81.5 mm
  1991-07-09  :  75.6 mm
  1991-07-10  :  39.5 mm
  1991-07-11  :  35.2 mm
  1991-07-12  :  47.7 mm
  1991-07-13  :  79.5 mm
  1991-07-14  :  54.1 mm
  1991-07-15  :  90.7 mm
  1991-07-16  :  118.9 mm
  1991-07-17  :  151.2 mm
  1991-07-18  :  96.8 mm
  1991-07-19  :  52.4 mm
  1991-07-20  :  50.8 mm
  1991-07-21  :  54.3 mm
  1991-07-22  :  61.4 mm
  1991-07-23  :  77.1 mm
  1991-07-24  :  50.4 mm
  1991-07-25  :  55.2 mm
  1991-07-26  :  97.3 mm
  1991-07-27  :  101.9 mm
  1991-07-28  :  82.1 mm
  1991-07-29  :  67.7 mm
  1991-07-30  :  63.0 mm
  1991-07-31  :  55.8 mm

July 1991 total : 1971.6 mm
July 1991 max   : 151.2 mm


In [12]:
# Loop through all yearly NC files and extract Goa daily rainfall

DATA_DIR = "imd_data/imd_rainfall-20260513T063252Z-3-001/imd_rainfall/"

GOA_LAT_MIN, GOA_LAT_MAX = 14.9, 15.7
GOA_LON_MIN, GOA_LON_MAX = 73.7, 74.3

all_years = []

# Only pick files that have a 4-digit year in the name
# This skips the unnamed files like RF25_ind_rfp25.nc
import re
import os

if not os.path.isdir(DATA_DIR):
    print(f"Error: Data directory '{DATA_DIR}' not found.")
    print("Please ensure you have run the file upload and unzipping cells (specifically 9l68j2X_X6LE and AXOmzZWtaE_4) successfully.")
    print("No files will be processed until the data directory is available.")
    files = [] # Initialize files as empty to prevent further errors
else:
    files = sorted(os.listdir(DATA_DIR))
    files = [f for f in files if re.match(r'RF25_ind\d{4}_rfp25\.nc$', f)]

    print(f"Files to process: {len(files)}")
    print("Years found:", [re.search(r'\d{4}', f).group() for f in files])

Files to process: 34
Years found: ['1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


In [16]:
# Loop through all 34 files and extract Goa daily rainfall

daily_list = []

import re
import os
import xarray as xr
import pandas as pd

# Define data directory and Goa bounding box (copied from previous cells to ensure self-containment)
DATA_DIR = "imd_data/imd_rainfall-20260513T063252Z-3-001/imd_rainfall/"
GOA_LAT_MIN, GOA_LAT_MAX = 14.9, 15.7
GOA_LON_MIN, GOA_LON_MAX = 73.7, 74.3

# Ensure 'files' is defined, copied from previous cell EixcvzJKcx3e
if not os.path.isdir(DATA_DIR):
    print(f"Error: Data directory '{DATA_DIR}' not found.")
    print("Please ensure you have run the file upload and unzipping cells (specifically 9l68j2X_6LE and AXOmzZWtaE_4) successfully.")
    print("No files will be processed until the data directory is available.")
    files = [] # Initialize files as empty to prevent further errors
else:
    files = sorted(os.listdir(DATA_DIR))
    # Corrected regex: use single backslashes for special characters
    files = [f for f in files if re.match(r'RF25_ind\d{4}_rfp25\.nc$', f)]

# Optional: print for debugging, similar to EixcvzJKcx3e's original output
# if files:
#     print(f"Files to process: {len(files)}")
#     print("Years found:", [re.search(r'\d{4}', f).group() for f in files])

for fname in files:
    year = re.search(r'\d{4}', fname).group()
    ds_yr = xr.open_dataset(DATA_DIR + fname)

    # Extract Goa region
    goa_yr = ds_yr['RAINFALL'].sel(
        LATITUDE  = slice(GOA_LAT_MIN, GOA_LAT_MAX),
        LONGITUDE = slice(GOA_LON_MIN, GOA_LON_MAX)
    )

    # Spatial average across Goa grid points
    goa_daily_yr = goa_yr.mean(dim=['LATITUDE', 'LONGITUDE'], skipna=True)

    # Convert to pandas Series
    times = pd.DatetimeIndex(ds_yr['TIME'].values)
    series = pd.Series(goa_daily_yr.values, index=times, name='rainfall_mm')

    daily_list.append(series)
    ds_yr.close()
    print(f"  {year} : {len(series)} days | total = {series.sum():.0f} mm")

# Combine all years into one Series
daily_goa = pd.concat(daily_list).sort_index()
print(f"\nFull series: {daily_goa.index[0].date()} to {daily_goa.index[-1].date()}")
print(f"Total days : {len(daily_goa)}")

  1991 : 365 days | total = 4232 mm
  1992 : 366 days | total = 4280 mm
  1993 : 365 days | total = 4471 mm
  1994 : 365 days | total = 6128 mm
  1995 : 365 days | total = 4008 mm
  1996 : 366 days | total = 4063 mm
  1997 : 365 days | total = 4996 mm
  1998 : 365 days | total = 3898 mm
  1999 : 365 days | total = 4598 mm
  2000 : 366 days | total = 4071 mm
  2001 : 365 days | total = 3157 mm
  2002 : 365 days | total = 3073 mm
  2003 : 365 days | total = 3503 mm
  2004 : 366 days | total = 3476 mm
  2006 : 365 days | total = 5863 mm
  2007 : 365 days | total = 4756 mm
  2008 : 366 days | total = 3987 mm
  2009 : 365 days | total = 4097 mm
  2010 : 365 days | total = 3969 mm
  2011 : 365 days | total = 4176 mm
  2012 : 366 days | total = 3255 mm
  2013 : 365 days | total = 3887 mm
  2014 : 365 days | total = 3680 mm
  2015 : 365 days | total = 2510 mm
  2016 : 366 days | total = 2731 mm
  2017 : 365 days | total = 2673 mm
  2018 : 365 days | total = 3190 mm
  2019 : 365 days | total = 

In [17]:
# Convert daily to monthly totals
monthly_goa = daily_goa.resample('MS').sum()

# 2005 is missing — fill with NaN then interpolate
monthly_goa['2005'] = np.nan
monthly_goa = monthly_goa.interpolate(method='linear')

print(f"Monthly series: {len(monthly_goa)} months")
print(f"Range: {monthly_goa.index[0].strftime('%b %Y')} to {monthly_goa.index[-1].strftime('%b %Y')}")
print(f"NaN remaining: {monthly_goa.isna().sum()}")
print(f"\nSample — first 6 months of 1991:")
print(monthly_goa.head(6).round(1))
print(f"\n2005 interpolated values:")
print(monthly_goa['2005'].round(1))

Monthly series: 420 months
Range: Jan 1991 to Dec 2025
NaN remaining: 0

Sample — first 6 months of 1991:
1991-01-01      0.000000
1991-02-01      0.000000
1991-03-01      0.000000
1991-04-01     45.599998
1991-05-01     59.900002
1991-06-01    934.500000
Freq: MS, Name: rainfall_mm, dtype: float32

2005 interpolated values:
2005-01-01    0.0
2005-02-01    0.0
2005-03-01    0.0
2005-04-01    0.0
2005-05-01    0.0
2005-06-01    0.0
2005-07-01    0.0
2005-08-01    0.0
2005-09-01    0.0
2005-10-01    0.0
2005-11-01    0.0
2005-12-01    0.0
Freq: MS, Name: rainfall_mm, dtype: float32


In [18]:
# Fix 2005 using average of same month in 2004 and 2006
for month in range(1, 13):
    val_2004 = monthly_goa[f'2004-{month:02d}-01']
    val_2006 = monthly_goa[f'2006-{month:02d}-01']
    monthly_goa[f'2005-{month:02d}-01'] = (val_2004 + val_2006) / 2

print("2005 after climatological fix:")
for idx, val in monthly_goa['2005'].items():
    print(f"  {idx.strftime('%b %Y')} : {val:.1f} mm")

print(f"\nAnnual total 2005 (estimated): {monthly_goa['2005'].sum():.0f} mm")
print(f"Annual total 2004 (actual)   : {monthly_goa['2004'].sum():.0f} mm")
print(f"Annual total 2006 (actual)   : {monthly_goa['2006'].sum():.0f} mm")

2005 after climatological fix:
  Jan 2005 : 0.0 mm
  Feb 2005 : 0.0 mm
  Mar 2005 : 8.4 mm
  Apr 2005 : 1.8 mm
  May 2005 : 215.7 mm
  Jun 2005 : 989.7 mm
  Jul 2005 : 1619.8 mm
  Aug 2005 : 1327.3 mm
  Sep 2005 : 343.8 mm
  Oct 2005 : 148.1 mm
  Nov 2005 : 14.9 mm
  Dec 2005 : 0.0 mm

Annual total 2005 (estimated): 4670 mm
Annual total 2004 (actual)   : 3476 mm
Annual total 2006 (actual)   : 5863 mm


In [19]:
# Save clean monthly series to CSV
monthly_goa.to_csv('goa_monthly_rainfall_clean.csv', header=True)

# Verify the saved file
df_check = pd.read_csv('goa_monthly_rainfall_clean.csv', index_col=0, parse_dates=True)
print(f"Saved file shape : {df_check.shape}")
print(f"Columns          : {list(df_check.columns)}")
print(f"Date range       : {df_check.index[0].strftime('%b %Y')} to {df_check.index[-1].strftime('%b %Y')}")
print(f"NaN values       : {df_check.isna().sum().sum()}")
print(f"\nFirst 3 rows:")
print(df_check.head(3).round(2))
print(f"\nLast 3 rows:")
print(df_check.tail(3).round(2))

Saved file shape : (420, 1)
Columns          : ['rainfall_mm']
Date range       : Jan 1991 to Dec 2025
NaN values       : 0

First 3 rows:
            rainfall_mm
1991-01-01          0.0
1991-02-01          0.0
1991-03-01          0.0

Last 3 rows:
            rainfall_mm
2025-10-01       285.13
2025-11-01        29.93
2025-12-01         0.00
